In [131]:
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
import numpy as np

In [132]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

con = sqlite3.connect('C:\\Users\\muhju\\Downloads\\sem 3 ojt\\proj 3.1\\data\\raw\\ipl.db')

def q(sql):
    return pd.read_sql_query(sql, con)

In [133]:
q(""" SELECT COUNT(*) AS n,
        SUM(CASE WHEN toss_winner = match_winner
                 THEN 1 ELSE 0 END)AS k
FROM matches_clean
WHERE result='win';""")

,n,k
0,1187,613


syntax= stats.binomtest(k,n,p)
k=no of success
n= total trail,data
p=expected probability



In [134]:
# if pvalue<sv then significant other wise not significant
p=stats.binomtest(613,1187,0.5).pvalue
ci=proportion_confint(613,1187, method='wilson')
print(p)
print(ci)

0.2700389712415702
(0.48799203833275506, 0.5447579131499739)


Since the p-value (0.2700) is greater than the significance level (0.05), we fail to reject the null hypothesis. There is insufficient statistical evidence to conclude that the true population proportion is different from 50% in the analyzed data.

In [135]:
#Does the decision taken after the toss matter?
#h0- Captains who choose to bat and captains who 
# choose to field win equally often. The gap we saw is luck.
#h1-The two choices do not win equally often.

q(""" SELECT toss_decision,
        SUM(CASE WHEN toss_winner = match_winner
                 THEN 1 ELSE 0 END ) AS won,
        SUM(CASE WHEN toss_winner <> match_winner
                 THEN 1 ELSE 0 END) AS lost
FROM matches_clean
WHERE result = 'win'
GROUP BY toss_decision;""")

,toss_decision,won,lost
0,bat,184,218
1,field,429,356


In [136]:
# if pvalue<sv then significant other wise not significant
table =[[184,218],[429,356]]
chi2, p, dof, expected = stats.chi2_contingency(table)
print(round(chi2,2),round(p,4))

8.04 0.0046


Since the p-value  is less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence of an association between the toss decision and the match outcome in the analyzed data.

In [137]:
#Does the chasing team really have an advantage?
#h0-Both sides win half the time. The 54.51% we saw is luck.
#h1-A chase does not succeed half the time.

q(""" SELECT COUNT(*) AS n,
        SUM(chase_won) AS k
FROM v_match_totals;""")

,n,k
0,1187,647


In [138]:
r=stats.binomtest(647,1187,0.5).pvalue
ri=proportion_confint(647,1187, method='wilson')
print(r)
print(ri)

0.002080449027018435
(0.5166432360570298, 0.5732091949340473)


Since the p-value is less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence that the chasing success rate is different from 50% in the analyzed data. The 95% Wilson confidence interval (51.66%–57.31%) further supports this finding.

In [139]:
#Does the chase win rate really fall as the target rises?

# h0-The chase win rate is the same whatever the target. 
# The differences between the bands are luck.

#h1- The chase win rate is not the same in every target band.

q(""" SELECT CASE WHEN first_innings_runs<140 THEN 'under 140'
             WHEN first_innings_runs <160 THEN '140-159'
             WHEN first_innings_runs <180 THEN '160-179'
             WHEN first_innings_runs <200 THEN '180-199'
             ELSE '200+' END AS band,
        SUM(chase_won) AS won,
        SUM(1-chase_won) AS lost
FROM  v_match_totals
GROUP BY band;""")

,band,won,lost
0,140-159,171,70
1,160-179,166,146
2,180-199,83,134
3,200+,46,152
4,under 140,181,38


In [140]:
table =[[171,70],[166,146],[83,134],[46,152],[181,38]]
chi2, p, dof, expected = stats.chi2_contingency(table)
print(round(chi2,2))
print(p)

197.68
1.1854224475712924e-41


Since the p-value (1.1854 × 10⁻⁴¹) is far less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence that the chase win rate differs across the target bands in the analyzed data. The results show that the chase win rate decreases substantially as the target increases.

In [141]:
#  are defended totals actually bigger than chased ones?
# h0 both kinds of total average the same . the gap we see is luck
# h1 the two averages are not equal.

q(""" SELECT first_innings_runs , chase_won
FROM v_match_totals;""")

,first_innings_runs,chase_won
0,222,0
1,207,0
2,184,1
3,183,1
4,163,1
...,...,...
1182,75,1
1183,222,1
1184,243,1
1185,155,1


In [142]:
df=pd.read_sql("""SELECT first_innings_runs, chase_won
FROM   v_match_totals;""",con)
defended = df[df.chase_won == 0].first_innings_runs
chased   = df[df.chase_won == 1].first_innings_runs
t, p = stats.ttest_ind(defended, chased, equal_var=False)
# move 6 needs the effect size too
pooled = np.sqrt((defended.std()**2 + chased.std()**2) / 2)
d = (defended.mean() - chased.mean()) / pooled
print(d)
print(pooled)
print(t)
print(p)

0.9162898870801979
30.342879534020167
15.730238965353694
1.17272268263676e-50


Since the p-value (1.1727 × 10⁻⁵⁰) is far less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence that the average first-innings score differs between successfully chased and defended matches. The Cohen's \(d\) value of 0.9163 indicates a large effect size, suggesting that the difference is also practically substantial.

In [143]:
# is the powerplay really faster than the middle overs?
# claim: teams score faster in the first
# six overs than they do in overs 7 to 15.
#h0= Both phases score at the same rate.
# Thegap we saw is luck.
#h1=The two run rates are not equal.

df = pd.read_sql("""SELECT match_id, innings,
       6.0 * SUM(CASE WHEN phase='Powerplay' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) AS pp,

       6.0 * SUM(CASE WHEN phase='Middle' THEN total_runs ELSE 0 END)
       / SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) AS mid

FROM v_ball

WHERE innings IN (1,2)

GROUP BY match_id, innings

HAVING
    SUM(CASE WHEN phase='Powerplay' THEN is_legal ELSE 0 END) > 0
    AND
    SUM(CASE WHEN phase='Middle' THEN is_legal ELSE 0 END) > 0""", con)
df

,match_id,innings,pp,mid
0,335982,1,10.166667,10.333333
1,335982,2,4.333333,6.111111
2,335983,1,8.833333,12.000000
3,335983,2,10.500000,11.333333
4,335984,1,6.666667,6.222222
...,...,...,...,...
2401,1529284,2,15.333333,11.222222
2402,1529285,1,9.833333,8.111111
2403,1529285,2,11.500000,8.444444
2404,1529286,1,9.333333,10.111111


In [144]:
t,p = stats.ttest_rel(df.pp, df.mid)
print(round(t,2),round(p,4))

1.53 0.1259


Since the p-value (0.1259) is greater than the significance level (0.05), we fail to reject the null hypothesis. There is insufficient statistical evidence to conclude that there is a significant difference between the pp and mid measurements in the analyzed data.

In [145]:
#Do left-handers really hit harder than right-handers?
#claim:Left-handed batters have a higher strike rate than right-handed batters.
#h0=Both hit at the same rate. The gap found is luck.
# h1=The two average strike rates are not equal.

df=pd.read_sql("""
SELECT batter,batsman_type,
       100.0*SUM(batter_runs)/SUM(is_legal) AS sr,
       SUM(is_legal) AS balls
FROM v_ball
WHERE batsman_type IS NOT NULL
GROUP BY batter,batsman_type
HAVING balls>500;""",con)
df

,batter,batsman_type,sr,balls
0,A Badoni,Right hand Bat,139.263804,815
1,A Symonds,Right hand Bat,130.738255,745
2,AB de Villiers,Right hand Bat,152.181604,3392
3,AC Gilchrist,Left hand Bat,139.514498,1483
4,AD Mathews,Right hand Bat,126.352531,573
...,...,...,...,...
132,Washington Sundar,Left hand Bat,130.518234,521
133,Y Venugopal Rao,Right hand Bat,118.389423,832
134,YBK Jaiswal,Left hand Bat,154.104478,1608
135,YK Pathan,Right hand Bat,143.741588,2229


In [146]:
left=df[df.batsman_type=='Left hand Bat']
right=df[df.batsman_type=='Right hand Bat']
t,p = stats.ttest_ind(left.sr, right.sr, equal_var=False)
print(t)
print(p)

-0.007335404976981389
0.994160816878741


Since the p-value (0.9942) is greater than the significance level (0.05), we fail to reject the null hypothesis. There is insufficient statistical evidence to conclude that the average strike rate differs significantly between left-hand and right-hand batsmen in the analyzed data.

In [147]:
# are spinners genuinely cheaper than fast bowlers?
# CLAIM: spin bowlers concede fewer runs per ball than fast bowlers.
#h0=Both concede at the same rate. The gap we saw is luck.
#h1=The two average economy rates are not equal.

df=pd.read_sql("""
              SELECT bowler,
              CASE WHEN bowler_type LIKE '%Legbreak%'
                   OR bowler_type LIKE '%Offbreak%'
                   OR bowler_type LIKE '%Orthodox%'
                   OR bowler_type LIKE '%Googly%'
                   OR bowler_type LIKE '%Wrist spin%'
                THEN 'Spin' ELSE 'Pace' END AS style,
              6.0*SUM(bowler_runs)/SUM(is_legal) AS economy,
              SUM(is_legal) AS balls
FROM v_ball
WHERE trim(coalesce(bowler_type,'')) <> ''
GROUP BY bowler, style
HAVING balls >= 500;""",con)
df
 

,bowler,style,economy,balls
0,A Kumble,Spin,6.578238,965
1,A Mishra,Spin,7.377633,3371
2,A Nehra,Pace,7.845912,1908
3,A Nortje,Pace,9.085714,1120
4,A Symonds,Spin,7.673624,527
...,...,...,...,...
144,YK Pathan,Spin,7.401918,1147
145,YS Chahal,Spin,8.027912,3941
146,Yash Dayal,Pace,9.575201,871
147,Yuvraj Singh,Spin,7.436133,869


In [148]:
spin = df[df['style']=='Spin']
pace = df[df['style']=='Pace']
t,p = stats.ttest_ind(spin.economy, pace.economy, equal_var=False)
print(t)
print(p)

-7.075597311698715
1.1249651433007181e-10


In [149]:
pooled = np.sqrt((spin.economy.std()**2 + pace.economy.std()**2) / 2)
d = (pace.economy.mean() - spin.economy.mean()) / pooled
print(d)
#0.2=small effect size
#0.5=medium effect size
#0.8=large effect size  
#d= effect size --- means how big the change is cohens d

1.1866831126238373


Since the p-value (8.4256 × 10⁻¹²) is far less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence that the average economy rate differs between spin and pace bowlers in the analyzed data.

In [150]:
# Does Sawai Mansingh really favour the chasing team?
# Chasing works better at Sawai Mansingh Stadium than it does elsewhere.

# h0= Chasing at Sawai Mansingh works exactly as well as chasing anywhere else— 54.51% 
# — and the 65.15% we saw is luck.
# h1=The chase rate at this ground is not 54.51%.

df=pd.read_sql("""SELECT COUNT(*) AS n,
        SUM(chase_won) AS k
FROM v_match_totals
WHERE venue_clean='Sawai Mansingh Stadium';""",con)
df

,n,k
0,66,43


In [151]:
#overall chasing rate = 54.51%
p=stats.binomtest(43,66,0.5451).pvalue
ci=proportion_confint(43,66, method='wilson')
print(p)
print(ci)

0.08493446917661332
(0.5311216021303434, 0.7552412593807046)


Since the p-value (0.0849) is greater than the significance level (0.05), we fail to reject the null hypothesis. There is insufficient statistical evidence to conclude that the chasing rate of 65.15% (43 out of 66) differs significantly from the overall chasing rate of 54.51% in the analyzed data. The 95% Wilson confidence interval (53.11%–75.52%) also contains the overall rate of 54.51%.

In [152]:
#Has scoring genuinely risen across the seasons?
#Claim:Average innings scores have risen over the nineteen seasons.
#h0-Average scores wander up and down with no trend, and the climb we saw is luck.
#h1-The season and the average score are related.

df=pd.read_sql("""SELECT m.season_year, AVG(first_i.runs) AS avg_score
FROM matches_clean m
JOIN v_innings first_i ON m.match_id = first_i.match_id
GROUP BY m.season_year;""",con)
df

,season_year,avg_score
0,2008,154.629310
1,2009,143.157895
2,2010,157.200000
3,2011,145.889655
4,2012,151.709459
5,2013,148.296053
6,2014,157.575000
7,2015,156.683761
8,2016,157.183333
9,2017,159.059322


In [153]:
# correlation coefficient
# -1 to +1
# 0 = no correlation, -1 =  negative correlation, +1 = positive correlation

r,p = stats.pearsonr(df.season_year, df.avg_score)
r_squared=r ** 2
print(r)
print(p)
print(r_squared)


0.8621194626048267
2.079722060152857e-06
0.7432499678020352


Since the p-value (2.0797 × 10⁻⁶) is less than the significance level (0.05), we reject the null hypothesis. There is statistically significant evidence of a strong positive correlation between season year and average score. The Pearson correlation coefficient (r = 0.8621) indicates a strong positive relationship, while R² = 0.7432 suggests that approximately 74.32% of the variation in average score is explained by the linear relationship with season year.

In [157]:
# Is Mumbai's head-to-head record over Chennai real?
# Claim :Mumbai Indians have  the better of Chennai Super Kings.
# h0-The two teams are evenly matched, and the 21-19 record is luck.
# h1-Mumbai do not win half of the meetings.
df=pd.read_sql("""SELECT COUNT(*) AS n,
       SUM(CASE WHEN match_winner = 'Mumbai Indians' 
                THEN 1 ELSE 0 END) AS k
FROM matches_clean
WHERE result='win' 
 AND ((team1='Mumbai Indians' 
      AND team2='Chennai Super Kings') 
 OR(team1='Chennai Super Kings'
   AND team2='Mumbai Indians'));""",con)
df

,n,k
0,40,21


In [158]:
p=stats.binomtest(21,40,0.5).pvalue
ci=proportion_confint(21,40, method='wilson')
print(p)
print(ci)

0.8746293123804207
(0.37497362106692467, 0.6706452988732111)


Since the p-value (0.8746) is greater than the significance level (0.05), we fail to reject the null hypothesis. There is insufficient statistical evidence to conclude that the observed proportion differs significantly from the hypothesized proportion. The 95% Wilson confidence interval ranges from 37.50% to 67.06%.

In [161]:
# Does the chasing advantage change from season to season?
# claim:Some seasons favour the chasing team much more than others.
# h0-The chase win rate is the same in every season, and the differences are luck.
# h1=The chase win rate is not the same in every season.
df=pd.read_sql(""" SELECT season_year,
       SUM(chase_won) AS wins,
       SUM(1-chase_won) AS lost
FROM v_match_totals
GROUP BY season_year;""",con)
df

,season_year,wins,lost
0,2008,36,22
1,2009,30,26
2,2010,28,31
3,2011,40,32
4,2012,40,34
5,2013,37,37
6,2014,37,22
7,2015,24,32
8,2016,41,19
9,2017,32,26


In [162]:
chi2, p, dof, expected = stats.chi2_contingency(df[['wins','lost']])
print(round(chi2,2))
print(round(p,4))

20.34
0.3141


Since the p-value (0.3141) is greater than the significance level (0.05), we fail to reject the null hypothesis. There is insufficient statistical evidence to conclude that there is a significant association between the variables represented in the contingency table